## 1. Import Library & Upload File dari Lokal

In [31]:
import io, os
import datetime
import numpy as np
import pandas as pd
from google.colab import files

uploaded = files.upload()
nama_file = list(uploaded.keys())[0]
print("File terupload:", nama_file)

df_raw = pd.read_excel(io.BytesIO(uploaded[nama_file]))
df = df_raw.copy()
df.head(10)

Saving Data sampah dilingkungan kampus tahun 2026.xlsx to Data sampah dilingkungan kampus tahun 2026 (1).xlsx
File terupload: Data sampah dilingkungan kampus tahun 2026 (1).xlsx


,ID,Tanggal,Lokasi,Jenis Sampah,Berat (kg),Jumlah,Kondisi,Cuaca,Aktivitas
0,1,20/09/2026,Gedung A,Plastik,2.5,35.0,Penuh,Cerah,Kuliah
1,2,20-09-2026,gedung a,Plastik,2.8,41.0,penuh,cerah,Kuliah
2,3,21/09/2026,Kantin,Organik,NaN,56.0,Meluber,Hujan,Istirahat
3,4,21/09/2026,Kantin,Organik,150.0,60.0,Penuh,Hujan,Istirahat
4,5,22/09/2026,Parkiran,Plastik,1.7,28.0,Sedang,Mendung,Sepi
5,6,22/09/2026,Parkiran,Plastik,1.7,28.0,Sedang,Mendung,Sepi
6,7,23/09/2026,Gedung B,Kertas,-2.0,19.0,Kosong,Cerah,Kuliah
7,8,23/09/2026,Gedung B,Kertas,3.2,NaN,Sedang,cerah,Kuliah
8,9,24/09/2026,Lapangan,Botol Plastik,4.5,72.0,Penuh,Cerah,Acara
9,10,24/09/2026,lapangan,Botol Plastik,4.5,72.0,penuh,Cerah,Acara


## 2. Eksplorasi Awal (Cek Kondisi Data Kotor)

In [32]:
print("Ukuran data:", df.shape)
print("\n--- Tipe data ---")
print(df.dtypes)
print("\n--- Missing value per kolom ---")
print(df.isna().sum())
print("\n--- Statistik numerik ---")
print(df[['Berat (kg)', 'Jumlah']].describe())

Ukuran data: (40, 9)

--- Tipe data ---
ID                int64
Tanggal          object
Lokasi           object
Jenis Sampah     object
Berat (kg)      float64
Jumlah          float64
Kondisi          object
Cuaca            object
Aktivitas        object
dtype: object

--- Missing value per kolom ---
ID              0
Tanggal         0
Lokasi          0
Jenis Sampah    0
Berat (kg)      3
Jumlah          2
Kondisi         0
Cuaca           0
Aktivitas       0
dtype: int64

--- Statistik numerik ---
       Berat (kg)      Jumlah
count   37.000000   38.000000
mean    13.186486   57.578947
std     39.738431   40.379036
min     -2.000000   -5.000000
25%      2.300000   25.750000
50%      3.800000   49.000000
75%      6.200000   86.750000
max    200.000000  150.000000


## 3. Cek Inkonsistensi pada Kolom Kategorikal

In [33]:
kolom_kategori = ['Lokasi', 'Jenis Sampah', 'Kondisi', 'Cuaca', 'Aktivitas']
for k in kolom_kategori:
    print(f"--- {k} ---")
    print(df[k].value_counts().to_string(), "\n")

--- Lokasi ---
Lokasi
Kantin      8
Parkiran    8
Gedung B    6
Gedung A    5
Lapangan    5
Gedung C    4
kantin      2
gedung a    1
lapangan    1 

--- Jenis Sampah ---
Jenis Sampah
Plastik          12
Kertas           10
Organik           9
Botol Plastik     4
Logam             4
organik           1 

--- Kondisi ---
Kondisi
Penuh      13
Sedang     10
Meluber     7
penuh       4
Kosong      4
meluber     1
kosong      1 

--- Cuaca ---
Cuaca
Cerah      17
Hujan       8
Mendung     8
cerah       3
mendung     2
hujan       2 

--- Aktivitas ---
Aktivitas
Kuliah       14
Istirahat    10
Sepi          8
Acara         8 



## 4. Cleansing Kolom Tanggal

In [34]:
def parse_tanggal(x):
    if isinstance(x, datetime.date):
        return pd.Timestamp(x)
    teks = str(x).strip().replace('-', '/')
    return pd.to_datetime(teks, format='%d/%m/%Y', errors='coerce')

df['Tanggal'] = pd.to_datetime(df['Tanggal'].apply(parse_tanggal))

print("Tanggal gagal terparse:", df['Tanggal'].isna().sum())
print("Rentang tanggal:", df['Tanggal'].min().date(), "s/d", df['Tanggal'].max().date())
print(df['Tanggal'].dtype)

Tanggal gagal terparse: 0
Rentang tanggal: 2026-09-20 s/d 2026-10-09
datetime64[ns]


## 5. Standarisasi Teks (Case & Spasi)

In [35]:
for k in ['Lokasi', 'Jenis Sampah', 'Kondisi', 'Cuaca', 'Aktivitas']:
    df[k] = df[k].astype(str).str.st rip().str.title()

GABUNG_BOTOL_PLASTIK = True
if GABUNG_BOTOL_PLASTIK:
    df['Jenis Sampah'] = df['Jenis Sampah'].replace({'Botol Plastik': 'Plastik'})

for k in ['Lokasi', 'Jenis Sampah', 'Kondisi', 'Cuaca']:
    print(f"--- {k} ---")
    print(df[k].value_counts().to_string(), "\n")

--- Lokasi ---
Lokasi
Kantin      10
Parkiran     8
Gedung A     6
Gedung B     6
Lapangan     6
Gedung C     4 

--- Jenis Sampah ---
Jenis Sampah
Plastik    16
Organik    10
Kertas     10
Logam       4 

--- Kondisi ---
Kondisi
Penuh      17
Sedang     10
Meluber     8
Kosong      5 

--- Cuaca ---
Cuaca
Cerah      20
Hujan      10
Mendung    10 



## 6. Hapus Data Duplikat

In [36]:
kolom_cek = [c for c in df.columns if c != 'ID']   # ID selalu unik, jadi diabaikan
dupe_mask = df.duplicated(subset=kolom_cek, keep='first')

print("Jumlah baris duplikat:", dupe_mask.sum())
print("ID yang dihapus:", df.loc[dupe_mask, 'ID'].tolist())

df = df[~dupe_mask].reset_index(drop=True)
print("Ukuran data setelah hapus duplikat:", df.shape)

Jumlah baris duplikat: 4
ID yang dihapus: [6, 10, 24, 36]
Ukuran data setelah hapus duplikat: (36, 9)


## 7. Tangani Nilai Tidak Valid & Outlier (Kolom Numerik)

In [37]:
mask_berat_invalid = df['Berat (kg)'] <= 0
mask_jumlah_invalid = df['Jumlah'] < 0
print("Berat tidak valid (<=0):", df.loc[mask_berat_invalid, 'ID'].tolist())
print("Jumlah tidak valid (<0):", df.loc[mask_jumlah_invalid, 'ID'].tolist())
df.loc[mask_berat_invalid, 'Berat (kg)'] = np.nan
df.loc[mask_jumlah_invalid, 'Jumlah'] = np.nan

# 7b. Outlier dengan metode IQR
def batas_iqr(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for kol in ['Berat (kg)', 'Jumlah']:
    bawah, atas = batas_iqr(df[kol])
    mask_out = (df[kol] < bawah) | (df[kol] > atas)
    print(f"\n{kol}: batas normal = [{bawah:.2f}, {atas:.2f}]")
    print("Outlier ID:", df.loc[mask_out, 'ID'].tolist(), "| nilai:", df.loc[mask_out, kol].tolist())
    df.loc[mask_out, kol] = np.nan   # dijadikan NaN lalu diisi di langkah berikutnya

Berat tidak valid (<=0): [7, 29]
Jumlah tidak valid (<0): [38]

Berat (kg): batas normal = [-3.60, 13.20]
Outlier ID: [4, 37] | nilai: [150.0, 200.0]

Jumlah: batas normal = [-66.50, 185.50]
Outlier ID: [] | nilai: []


## 8. Tangani Missing Value (Imputasi Median per Jenis Sampah)

In [38]:
print("Missing sebelum imputasi:")
print(df[['Berat (kg)', 'Jumlah']].isna().sum(), "\n")
for kol in ['Berat (kg)', 'Jumlah']:
    df[kol] = df.groupby('Jenis Sampah')[kol].transform(lambda s: s.fillna(s.median()))
    df[kol] = df[kol].fillna(df[kol].median())
df['Berat (kg)'] = df['Berat (kg)'].round(1)
df['Jumlah'] = df['Jumlah'].round().astype(int)
print("Missing setelah imputasi:")
print(df[['Berat (kg)', 'Jumlah']].isna().sum())

Missing sebelum imputasi:
Berat (kg)    7
Jumlah        3
dtype: int64 

Missing setelah imputasi:
Berat (kg)    0
Jumlah        0
dtype: int64


## 9. Validasi Akhir (Sebelum vs Sesudah)

In [39]:
df = df.sort_values('Tanggal').reset_index(drop=True)

# Buang komponen jam (00:00:00) -> hanya tanggal (YYYY-MM-DD)
df['Tanggal'] = pd.to_datetime(df['Tanggal']).dt.date

print("Ukuran: sebelum", df_raw.shape, "-> sesudah", df.shape)
print("\nTipe data setelah cleansing:")
print(df.dtypes)
print("\nTotal missing :", df.isna().sum().sum())
print("Duplikat      :", df.duplicated(subset=[c for c in df.columns if c != 'ID']).sum())
print("Berat <= 0    :", (df['Berat (kg)'] <= 0).sum())
print("Jumlah < 0    :", (df['Jumlah'] < 0).sum())
print("\nStatistik numerik setelah cleansing:")
print(df[['Berat (kg)', 'Jumlah']].describe())
df

Ukuran: sebelum (40, 9) -> sesudah (36, 9)

Tipe data setelah cleansing:
ID                int64
Tanggal          object
Lokasi           object
Jenis Sampah     object
Berat (kg)      float64
Jumlah            int64
Kondisi          object
Cuaca            object
Aktivitas        object
dtype: object

Total missing : 0
Duplikat      : 0
Berat <= 0    : 0
Jumlah < 0    : 0

Statistik numerik setelah cleansing:
       Berat (kg)      Jumlah
count   36.000000   36.000000
mean     4.502778   61.972222
std      2.394456   39.224254
min      1.100000    8.000000
25%      2.600000   27.250000
50%      3.850000   58.000000
75%      6.800000   88.750000
max      9.500000  150.000000


,ID,Tanggal,Lokasi,Jenis Sampah,Berat (kg),Jumlah,Kondisi,Cuaca,Aktivitas
0,1,2026-09-20,Gedung A,Plastik,2.5,35,Penuh,Cerah,Kuliah
1,2,2026-09-20,Gedung A,Plastik,2.8,41,Penuh,Cerah,Kuliah
2,3,2026-09-21,Kantin,Organik,7.1,56,Meluber,Hujan,Istirahat
3,4,2026-09-21,Kantin,Organik,7.1,60,Penuh,Hujan,Istirahat
4,5,2026-09-22,Parkiran,Plastik,1.7,28,Sedang,Mendung,Sepi
5,7,2026-09-23,Gedung B,Kertas,2.6,19,Kosong,Cerah,Kuliah
6,8,2026-09-23,Gedung B,Kertas,3.2,23,Sedang,Cerah,Kuliah
7,9,2026-09-24,Lapangan,Plastik,4.5,72,Penuh,Cerah,Acara
8,11,2026-09-25,Gedung C,Organik,5.1,83,Meluber,Hujan,Kuliah
9,12,2026-09-25,Gedung C,Organik,4.8,79,Meluber,Hujan,Kuliah


## 10. Simpan Hasil ke Google Drive

In [40]:
from google.colab import drive
drive.mount('/content/drive')

folder_tujuan = '/content/drive/MyDrive/Data-Sampah-Bersih'
os.makedirs(folder_tujuan, exist_ok=True)

path_xlsx = os.path.join(folder_tujuan, 'Data_Sampah_Kampus_2026_Bersih.xlsx')

with pd.ExcelWriter(path_xlsx, engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='Data Bersih')
    ws = writer.sheets['Data Bersih']
    for sel in ws['B'][1:]:
        sel.number_format = 'DD/MM/YYYY'
df.to_csv(path_csv, index=False)

print("Tersimpan di Google Drive:")
print(" -", path_xlsx)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tersimpan di Google Drive:
 - /content/drive/MyDrive/Data-Sampah-Bersih/Data_Sampah_Kampus_2026_Bersih.xlsx
 - /content/drive/MyDrive/Data_Sampah_Kampus/Data_Sampah_Kampus_2026_Bersih.csv
